# Lesson 05 — Building makemore Part 4: Becoming a Backprop Ninja

- **GitHub issue:** [#5](https://github.com/majorgilles/karpathy_ml_course/issues/5)
- **Video:** https://youtu.be/q8SA3rM6ckI
- **Lesson guide:** [../README.md](../README.md)
- **Transcript:** [../transcript.md](../transcript.md)

Use this notebook for exploratory follow-along work. Move reusable code to `../src/`, lightweight checks to `../tests/`, and representative outputs to `../artifacts/`.

## 1. Prepare the tensor and plotting tools

This lesson keeps the familiar two-layer character MLP but exposes every small operation in its forward pass. PyTorch supplies tensor operations and performs the reference backward pass so later manually derived gradients can be checked against autograd. The functional API and Matplotlib are imported now for compact loss checks and diagnostic plots added later in the lesson.

The durable goal is not to memorize derivative formulas in isolation. It is to connect each forward tensor to the local operation that created it, determine how the scalar loss depends on it, and propagate that dependence backward with the chain rule.


In [1]:
import matplotlib.pyplot as plt  # noqa: F401  # Used by later diagnostic plots.
import torch  # Tensor operations and autograd reference gradients.
import torch.nn.functional as F  # noqa: F401  # Used by later compact loss exercises.

%matplotlib inline


## 2. Configure the experiment in one place

This cell collects the data split, architecture, initialization, BatchNorm, and mini-batch settings used by the current notebook. Edit these values before running the remaining cells when you want to compare a different experiment. The vocabulary size is not a hyperparameter: it is derived later from the characters present in the dataset.

The defaults reproduce the current lesson setup. `TRAIN_END_FRACTION = 0.8` and `DEV_END_FRACTION = 0.9` are cumulative split boundaries, giving 80% train, 10% development, and the remaining 10% test. `PARAMETER_INIT_SCALE` deliberately makes several normally zero-initialized values small and nonzero so incorrect manual derivatives are harder to hide.

Changing dimensions also changes downstream tensor shapes and the total parameter count. The Markdown formulas below describe the displayed default configuration unless stated symbolically.


In [2]:
# Data split configuration.
DATA_SPLIT_SEED = 42
TRAIN_END_FRACTION = 0.8  # First 80% of shuffled complete names.
DEV_END_FRACTION = 0.9  # Next 10%; the remaining 10% becomes test data.

# Model architecture.
BLOCK_SIZE = 3  # Number of preceding token IDs in each context.
EMBEDDING_SIZE = 10  # Learned features per character.
HIDDEN_SIZE = 64  # Neurons in the single hidden layer.

# Reproducible parameter and mini-batch sampling.
MODEL_SEED = 2147483647
BATCH_SIZE = 32

# Initialization and normalization.
TANH_GAIN = 5 / 3
PARAMETER_INIT_SCALE = 0.1  # Deliberately nonzero for robust gradient checks.
BATCHNORM_GAIN_CENTER = 1.0  # Initialize gamma near the identity scale.
BATCHNORM_EPS = 1e-5  # Stabilize division by a very small variance.


## 3. Load the name sequences

Each line of `names.txt` is one complete training sequence such as `emma`. The path fallback supports running the notebook either from the repository root or from this notebook directory. At this stage the names remain strings; later cells convert each character transition into a numeric context-target training example.


In [3]:
# Load every name; each line in names.txt becomes one training sequence.
from pathlib import Path

candidate_paths = [
    Path("data/raw/names.txt"),  # Kernel launched from the repository root.
    Path("../../../data/raw/names.txt"),  # Kernel launched from this notebook folder.
]
names_path = next(path for path in candidate_paths if path.exists())
words = names_path.read_text(encoding="utf-8").splitlines()

words[:8]  # Inspect a small sample before building numeric examples.

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# Confirm how many complete name sequences are available.
len(words)

32033

## 4. Build the 27-token vocabulary

The vocabulary contains the 26 lowercase letters plus the shared boundary token `.`. `stoi` maps a character to its integer ID for tensor indexing, while `itos` reverses that mapping for interpretation and future sampling. Boundary-token ID `0` pads the beginning of a context and marks the end of a name.

Deriving `chars` from the dataset ensures that the mapping reflects the symbols actually present. Sorting makes letter IDs deterministic: `a` receives ID `1`, through `z` receiving ID `26`.


In [5]:
# Build the vocabulary and deterministic mappings between characters and integer IDs.
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0  # Boundary token: context padding and end-of-name target.
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)  # 26 letters plus `.` = 27 candidate next tokens.
print(itos)
print(vocab_size)


{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


## 5. Build context-target examples and split complete names

With the default `BLOCK_SIZE = 3`, every training example contains three preceding token IDs in `X` and one expected next-token ID in `Y`. For a name such as `emma`, the first context is `...` and its expected target is `e`; the context then shifts one position and includes each observed target. Appending `.` supplies the final end-of-name target.

Complete names are shuffled with `DATA_SPLIT_SEED` and divided 80/10/10 before examples are built. This keeps every transition from one name in exactly one split. Only `Xtr` and `Ytr` will supply parameter updates; development data is for model comparison, and test data remains reserved for final evaluation.

For a split containing `N` character transitions, the resulting shapes are `(N, BLOCK_SIZE)` contexts and `(N,)` expected targets. The printed sizes therefore count training examples, not complete names.


In [6]:
# Build aligned context-target examples for one complete-name split.
def build_dataset(words: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
    X, Y = [], []

    for w in words:  # Keep all transitions from this name in the same split.
        context = [0] * BLOCK_SIZE  # Begin with boundary-token padding.
        for ch in w + ".":  # Include the end-of-name boundary as a target.
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]  # Shift left and append the observed target.

    X = torch.tensor(X)  # (N, BLOCK_SIZE): context rows.
    Y = torch.tensor(Y)  # (N,): expected next-token IDs.
    print(X.shape, Y.shape)
    return X, Y


import random  # noqa: E402  # Keep the course's import location below the helper.

random.seed(DATA_SPLIT_SEED)
random.shuffle(words)
n1 = int(TRAIN_END_FRACTION * len(words))
n2 = int(DEV_END_FRACTION * len(words))

Xtr, Ytr = build_dataset(words[:n1])  # Training split: the only source of updates.
Xdev, Ydev = build_dataset(words[n1:n2])  # Development split for model comparison.
Xte, Yte = build_dataset(words[n2:])  # Test split reserved for final evaluation.


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


## 6. Compare manual gradients with autograd

Later exercises will construct a manual gradient `dt` for a forward tensor `t`. PyTorch stores its reference gradient in `t.grad` after `loss.backward()`. The `cmp` helper reports exact equality, approximate floating-point agreement, and the largest absolute elementwise difference.

Exact equality is stricter than mathematical correctness because algebraically equivalent formulas can perform floating-point operations in a different order. Approximate equality is therefore the main correctness check, while maximum difference shows the size of any disagreement. Both tensors being compared must have the same shape as the forward tensor whose gradient they represent.


In [7]:
# Compare a manually derived gradient `dt` with PyTorch's reference gradient for `t`.
def cmp(s: str, dt: torch.Tensor, t: torch.Tensor) -> None:
    ex = torch.all(dt == t.grad).item()  # Strict element-for-element equality.
    app = torch.allclose(dt, t.grad)  # Floating-point approximate equality.
    maxdiff = (dt - t.grad).abs().max().item()  # Largest disagreement.
    print(
        f"{s:15s} | exact: {str(ex):5s} | "
        f"approximate: {str(app):5s} | maxdiff: {maxdiff}"
    )


## 7. Initialize a deliberately testable MLP

The model uses a default `(27, 10)` embedding table, a 30-to-64 hidden linear layer, BatchNorm, `tanh`, and a 64-to-27 output layer. Three 10-feature embeddings produce the 30 hidden-layer inputs. The output layer returns one candidate score for each of the 27 possible next tokens.

`W1` uses the `tanh` gain `5/3` together with fan-in scaling:

$$
\operatorname{scale}(W_1)
=
\frac{5/3}{\sqrt{30}}
$$

Several parameters intentionally use `PARAMETER_INIT_SCALE` random values instead of conventional zeros. Nonzero values prevent symmetry or zero-valued terms from accidentally hiding an incorrect manual derivative. `b1` is also retained even though BatchNorm centering makes a pre-normalization hidden bias redundant; this creates another gradient that the manual implementation must handle correctly.

With the default dimensions, the seven trainable tensors contain 4,137 scalar values:

$$
27 \times 10
+
30 \times 64
+
64
+
64 \times 27
+
27
+
64
+
64
=
4{,}137
$$

Every scaled tensor is created before `requires_grad` is enabled so the entries in `parameters` remain leaf tensors and receive `.grad` values directly.


In [8]:
# Fixed generator makes parameters and the following mini-batch reproducible.
g = torch.Generator().manual_seed(MODEL_SEED)
C = torch.randn((vocab_size, EMBEDDING_SIZE), generator=g)

# Layer 1: BLOCK_SIZE embeddings become HIDDEN_SIZE hidden pre-activations.
W1 = (
    torch.randn((EMBEDDING_SIZE * BLOCK_SIZE, HIDDEN_SIZE), generator=g)
    * TANH_GAIN
    / ((EMBEDDING_SIZE * BLOCK_SIZE) ** 0.5)
)  # Tanh-gain and fan-in scaled hidden weights.
b1 = torch.randn(HIDDEN_SIZE, generator=g) * PARAMETER_INIT_SCALE

# Layer 2: one score for every vocabulary candidate.
W2 = (
    torch.randn((HIDDEN_SIZE, vocab_size), generator=g) * PARAMETER_INIT_SCALE
)
b2 = torch.randn(vocab_size, generator=g) * PARAMETER_INIT_SCALE

# BatchNorm affine parameters: one scale and shift per hidden neuron.
bngain = (
    torch.randn((1, HIDDEN_SIZE)) * PARAMETER_INIT_SCALE
    + BATCHNORM_GAIN_CENTER
)
bnbias = torch.randn((1, HIDDEN_SIZE)) * PARAMETER_INIT_SCALE

# Nonstandard nonzero values help expose incorrect manual backward formulas.
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True  # Populate each leaf parameter's `.grad` during backward.


4137


## 8. Sample one reproducible training mini-batch

The manual backward exercise operates on one deliberately supplied mini-batch rather than the full training dataset. `ix` samples `BATCH_SIZE = 32` training-row indices uniformly with replacement, producing `Xb` with shape `(32, 3)` and aligned expected targets `Yb` with shape `(32,)`.

With the defaults, the shorter name `n = BATCH_SIZE = 32` appears in the expanded mean, variance, and loss formulas. In this section, `n` means the number of training examples in this sampled batch; it is not the vocabulary size or the number of candidate tokens.


In [9]:
n = BATCH_SIZE  # Short name used in the expanded mean, variance, and loss formulas.

# Construct one reproducible mini-batch sampled from training rows with replacement.
ix = torch.randint(0, Xtr.shape[0], (BATCH_SIZE,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]  # Aligned contexts and expected next-token targets.


## 9. Expand the forward pass into differentiable sections

The following cells compute the same forward pass and scalar mini-batch loss as the original combined cell. They are separated only to make the computation graph easier to inspect: embedding and first linear layer, BatchNorm, nonlinearity and output layer, explicit cross-entropy, and the PyTorch reference backward pass.

Each intermediate tensor is still named because the next exercise will traverse these operations in reverse. A future name beginning with `d`, such as `dlogprobs`, means the derivative of the scalar batch loss with respect to that forward tensor:

$$
dT
=
\frac{\partial \mathcal{L}_{\mathrm{batch}}}{\partial T}
$$

Here, `T` can be an intermediate tensor or trainable parameter. Splitting the code into cells does not detach tensors from autograd; running the cells from top to bottom still creates one connected computation graph.


### 9.1 Embed the context and apply the first linear layer

For training example `i` and context position `k`, the integer token ID `Xb[i, k]` selects one row from embedding table `C`:

$$
e_{ik}
=
C_{X_{b,ik}}
$$

Each `e_ik` contains `EMBEDDING_SIZE = 10` learned features. Concatenating all `BLOCK_SIZE = 3` vectors forms one 30-feature row `x_i`:

$$
x_i
=
\operatorname{concat}\left(e_{i1}, e_{i2}, e_{i3}\right)
$$

The first affine layer then produces one raw value for each of 64 hidden neurons:

$$
a_i
=
x_i W_1 + b_1
$$

In code, `a` is named `hprebn` because it is the hidden pre-activation before BatchNorm. The default tensor path is:

```text
Xb (32, 3) → emb (32, 3, 10) → embcat (32, 30) → hprebn (32, 64)
```


In [29]:
# Look up and concatenate the three context embeddings for each example.
print(Xb.shape)
print(C.shape)
emb = C[Xb]  # (B, BLOCK_SIZE, EMBEDDING_SIZE)
print(emb.shape)
embcat = emb.view(emb.shape[0], -1)  # (B, BLOCK_SIZE * EMBEDDING_SIZE)
print(embcat.shape)

# First affine layer: one raw pre-activation per hidden neuron.
hprebn = embcat @ W1 + b1  # (B, HIDDEN_SIZE) "Hidden PRE BatchNorm => hprebn"


torch.Size([32, 3])
torch.Size([27, 10])
torch.Size([32, 3, 10])
torch.Size([32, 30])


### 9.2 Normalize each hidden neuron across the mini-batch

BatchNorm treats rows as training examples and columns as hidden neurons. For each neuron `j`, it first computes the mean of its `n = BATCH_SIZE = 32` raw pre-activations:

$$
\mu_j
=
\frac{1}{n}
\sum_{i=1}^{n} a_{ij}
$$

It centers every example by subtracting that neuron's mean:

$$
d_{ij}
=
a_{ij}-\mu_j
$$

It then computes the unbiased sample variance. Bessel's correction divides by `n - 1 = 31` rather than `n`:

$$
\sigma_j^2
=
\frac{1}{n-1}
\sum_{i=1}^{n} d_{ij}^2
$$

The stabilized inverse standard deviation adds `BATCHNORM_EPS` before applying the power `-1/2`:

$$
r_j
=
\left(\sigma_j^2 + \varepsilon\right)^{-1/2}
$$

Normalization and the learned affine transformation are:

$$
\widehat{a}_{ij}
=
d_{ij}r_j
$$

$$
\widetilde{a}_{ij}
=
\gamma_j\widehat{a}_{ij}+\beta_j
$$

Here, `i` identifies one training example, `j` identifies one hidden neuron, epsilon is `BATCHNORM_EPS`, gamma is `bngain`, and beta is `bnbias`. The `(1, HIDDEN_SIZE)` statistics and affine parameters broadcast across all `BATCH_SIZE` examples. The small operations remain separate because the later manual backward pass derives a gradient through each one.


In [11]:
# Compute one mini-batch mean and sample variance for each hidden neuron.
bnmeani = (1 / n) * hprebn.sum(0, keepdim=True)  # (1, HIDDEN_SIZE)
bndiff = hprebn - bnmeani  # (B, HIDDEN_SIZE): centered hidden values.
bndiff2 = bndiff**2  # (B, HIDDEN_SIZE): squared deviations.
# Bessel's correction divides the sample variance by n - 1 rather than n.
bnvar = (1 / (n - 1)) * bndiff2.sum(0, keepdim=True)  # (1, HIDDEN_SIZE)
bnvar_inv = (bnvar + BATCHNORM_EPS) ** -0.5  # Stabilized inverse std.

# Normalize, then apply one learned scale and shift per hidden neuron.
bnraw = bndiff * bnvar_inv  # (B, HIDDEN_SIZE): normalized values.
hpreact = bngain * bnraw + bnbias  # (B, HIDDEN_SIZE): BatchNorm output.


### 9.3 Apply `tanh` and compute vocabulary logits

The hidden nonlinearity applies `tanh` independently to every normalized and affine-transformed hidden value:

$$
h_{ij}
=
\tanh\left(\widetilde{a}_{ij}\right)
$$

This preserves the `(BATCH_SIZE, HIDDEN_SIZE)` shape while bounding every activation between `-1` and `1`. The second affine layer maps each example's 64 hidden activations to 27 vocabulary-candidate scores:

$$
z_i
=
h_i W_2+b_2
$$

`z_i` is the `logits` row for example `i`. Its 27 entries are unconstrained scores, not probabilities. The expected target `Yb[i]` is not used until the cross-entropy section.


In [12]:
# Hidden nonlinearity and output affine layer.
h = torch.tanh(hpreact)  # (B, HIDDEN_SIZE): activations bounded to (-1, 1).
logits = h @ W2 + b2  # (B, vocab_size): one score per candidate.


### 9.4 Expand softmax cross-entropy into atomic operations

For example `i`, let `z_ij` be the logit for vocabulary candidate `j`. Subtracting the largest logit from every score in the row prevents exponentiation from overflowing:

$$
m_i
=
\max_j z_{ij}
$$

$$
\widetilde{z}_{ij}
=
z_{ij}-m_i
$$

This shift does not change softmax because it multiplies every exponentiated score by the same row-wise constant. The code then creates positive unnormalized weights, sums all 27 candidates, and normalizes each row:

$$
c_{ij}
=
\exp\left(\widetilde{z}_{ij}\right)
$$

$$
s_i
=
\sum_{j=1}^{V} c_{ij}
$$

$$
p_{ij}
=
c_{ij}s_i^{-1}
$$

Here, `V = vocab_size = 27` is the number of vocabulary candidates. Every probability row sums to one. Taking logarithms gives `logprobs`, but only the entry indexed by the expected target `Yb[i]` contributes directly to example `i`'s negative log-likelihood:

$$
\mathcal{L}_i
=
-\log p_{i,Y_{b,i}}
$$

The scalar loss averages all `n = 32` supplied training examples:

$$
\mathcal{L}_{\mathrm{batch}}
=
-\frac{1}{n}
\sum_{i=1}^{n}
\log p_{i,Y_{b,i}}
$$

The other 26 candidates influence the selected probability through the shared denominator `s_i`. This expanded computation is mathematically equivalent to `F.cross_entropy(logits, Yb)`. The reciprocal remains written as `counts_sum**-1` because that operation order supports the later bit-exact gradient comparison.


In [25]:
# Expand cross-entropy into operations that can be differentiated separately.
logit_maxes = logits.max(1, keepdim=True).values # (B, 1): one maximum per row.
norm_logits = logits - logit_maxes  # Shift scores for numerical stability.
counts = norm_logits.exp()  # (B, vocab_size): positive unnormalized weights.
counts_sum = counts.sum(1, keepdims=True)  # (B, 1): row totals.
# The power form, rather than 1.0 / counts_sum, supports later bit-exact checks.
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv  # Candidate distributions summing to one.
logprobs = probs.log()  # Log-probability of every vocabulary candidate.
# Only the probability at each expected target index contributes directly to its NLL.
loss = -logprobs[range(n), Yb].mean()  # Average over B training examples.


### 9.5 Retain PyTorch's reference gradients

Before backpropagation, parameter gradients are reset to `None` so rerunning this section does not accumulate values from an earlier backward pass. Parameters are leaf tensors, so PyTorch retains their gradients automatically. The named forward intermediates are non-leaf tensors, so `retain_grad()` is required if their `.grad` fields will be inspected later.

Calling `loss.backward()` starts from the scalar batch loss and applies the chain rule through the connected graph in reverse order. For every retained tensor `T`, PyTorch then stores:

$$
T.\mathrm{grad}
=
\frac{\partial \mathcal{L}_{\mathrm{batch}}}{\partial T}
$$

These autograd values are reference answers only. The upcoming exercise will derive corresponding manual tensors such as `dlogprobs`, compare them with `T.grad` using `cmp`, and continue backward toward the parameters and embedding table. This section performs no parameter update and scores only the deliberately supplied training mini-batch.


In [15]:
# Clear parameter gradients so repeated runs do not accumulate reference values.
for p in parameters:
    p.grad = None

# Preserve gradients for every non-leaf intermediate used by the manual exercise.
for t in [
    logprobs,
    probs,
    counts,
    counts_sum,
    counts_sum_inv,
    norm_logits,
    logit_maxes,
    logits,
    h,
    hpreact,
    bnraw,
    bnvar_inv,
    bnvar,
    bndiff2,
    bndiff,
    hprebn,
    bnmeani,
    embcat,
    emb,
]:
    t.retain_grad()

loss.backward()  # Populate parameter and retained-intermediate reference gradients.
loss  # Display this supplied mini-batch's average NLL; no update occurs.


tensor(3.3480, grad_fn=<NegBackward0>)

In [17]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0 * 1 / n
cmp('logprobs', dlogprobs, logprobs)

dprobs = 1 / probs * dlogprobs
cmp('probs', dprobs, probs)

dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

#cmp('counts_sum', dcounts_sum, counts_sum)
#cmp('counts', dcounts, counts)
#cmp('norm_logits', dnorm_logits, norm_logits)
#cmp('logit_maxes', dlogit_maxes, logit_maxes)
#cmp('logits', dlogits, logits)
#cmp('h', dh, h)
#cmp('W2', dW2, W2)
#cmp('b2', db2, b2)
#cmp('hpreact', dhpreact, hpreact)
#cmp('bngain', dbngain, bngain)
#cmp('bnbias', dbnbias, bnbias)
#cmp('bnraw', dbnraw, bnraw)
#cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
#cmp('bnvar', dbnvar, bnvar)
#cmp('bndiff2', dbndiff2, bndiff2)
#cmp('bndiff', dbndiff, bndiff)
#cmp('bnmeani', dbnmeani, bnmeani)
#cmp('hprebn', dhprebn, hprebn)
#cmp('embcat', dembcat, embcat)
#cmp('W1', dW1, W1)
#cmp('b1', db1, b1)
#cmp('emb', demb, emb)
#cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
torch.Size([32, 1])
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
